# 텍스트 임베딩 (Text Embeddings)

**Skilljar Lesson L03 대응**

이 노트북에서 다루는 내용:
1. VoyageAI 임베딩 생성
2. 코사인 유사도 (Cosine Similarity) 계산
3. 유사도 기반 문서 검색
4. document vs query input_type 비교

In [ ]:
# ── Setup ──────────────────────────────────────────────
import numpy as np
import voyageai
from dotenv import load_dotenv

load_dotenv()

voyage_client = voyageai.Client()  # VOYAGE_API_KEY 환경변수 필요

## §1. 임베딩 생성 (Generating Embeddings)

텍스트를 고차원 벡터로 변환합니다.  
의미가 유사한 텍스트는 벡터 공간에서 **가까운 위치**에 배치됩니다.

In [ ]:
def generate_embedding(text: str, input_type: str = "document") -> list[float]:
    """텍스트의 임베딩 벡터를 생성한다.

    Args:
        text: 임베딩할 텍스트
        input_type: 'document' (인덱싱) 또는 'query' (검색)

    Returns:
        임베딩 벡터 (float 리스트)
    """
    result = voyage_client.embed(
        texts=[text],
        model="voyage-3",
        input_type=input_type
    )
    return result.embeddings[0]

In [ ]:
# 임베딩 생성 테스트
text = "콘크리트의 설계기준강도는 fck로 표기한다"
embedding = generate_embedding(text)

print(f"입력 텍스트: {text}")
print(f"임베딩 차원 수: {len(embedding)}")
print(f"처음 10개 값: {embedding[:10]}")
print(f"값 범위: [{min(embedding):.4f}, {max(embedding):.4f}]")

## §2. 코사인 유사도 (Cosine Similarity)

두 벡터가 얼마나 같은 방향을 가리키는지 측정합니다.  
값 범위: -1 (반대) ~ 0 (무관) ~ 1 (동일)

In [ ]:
def cosine_similarity(vec1: list[float], vec2: list[float]) -> float:
    """두 벡터의 코사인 유사도를 계산한다."""
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

In [ ]:
# 유사도 측정 테스트
texts = [
    "콘크리트 압축강도 기준",           # 쿼리
    "콘크리트의 설계기준강도는 fck로 표기한다",  # 관련 문서
    "concrete compressive strength",    # 영어 동의어
    "오늘 서울 날씨는 맑음이다",          # 무관 문서
    "철근의 항복강도 fy는 400 MPa이다",    # 부분 관련
]

# 모든 텍스트의 임베딩 생성
embeddings = [generate_embedding(t) for t in texts]

# 쿼리(첫 번째)와 나머지 문서의 유사도 계산
query_emb = embeddings[0]
print(f"쿼리: '{texts[0]}'\n")
for i in range(1, len(texts)):
    sim = cosine_similarity(query_emb, embeddings[i])
    print(f"  [{sim:.4f}] {texts[i]}")

## §3. 유사도 기반 문서 검색

여러 청크 중에서 쿼리와 가장 유사한 top-k를 찾습니다.

In [ ]:
# 샘플 청크 (S4_01에서 만든 구조 기반 청킹 결과를 시뮬레이션)
sample_chunks = [
    "이 기준은 건축물의 구조안전성을 확보하기 위한 최소한의 요구사항을 정한다.",
    "고정하중은 구조물 자체의 무게와 영구적으로 부착된 부분의 무게를 포함한다. 콘크리트의 단위중량은 24 kN/m3이다.",
    "적재하중은 건축물의 용도에 따라 다르게 적용한다. 주거용 건물의 바닥 적재하중은 2.0 kN/m2이다.",
    "콘크리트의 설계기준강도 fck는 최소 21 MPa 이상이어야 한다. 고강도 콘크리트의 경우 fck 40 MPa 이상을 적용할 수 있다.",
    "RC 보의 최소 철근비는 0.25*sqrt(fck)/fy 이상이어야 하며, 1.4/fy 이상이어야 한다.",
    "기둥의 최소 단면치수는 300mm 이상이어야 한다. 주근의 최소 개수는 4개이다.",
]

# 청크 임베딩 생성
chunk_embeddings = [generate_embedding(c, input_type="document") for c in sample_chunks]
print(f"{len(sample_chunks)}개 청크 임베딩 생성 완료")

In [ ]:
def search_chunks(query: str, chunks: list[str], chunk_embs: list, top_k: int = 3) -> list[dict]:
    """쿼리와 가장 유사한 청크를 검색한다."""
    query_emb = generate_embedding(query, input_type="query")

    similarities = [
        cosine_similarity(query_emb, emb) for emb in chunk_embs
    ]

    # 유사도 내림차순 정렬
    ranked = sorted(
        enumerate(similarities),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    return [{"text": chunks[i], "score": score} for i, score in ranked]


# 검색 테스트
query = "RC 보의 최소 철근비는?"
results = search_chunks(query, sample_chunks, chunk_embeddings, top_k=3)

print(f"쿼리: '{query}'\n")
for i, r in enumerate(results, 1):
    print(f"  {i}위 [{r['score']:.4f}] {r['text'][:80]}...")

## §4. document vs query input_type

VoyageAI는 `input_type`에 따라 임베딩 전략을 최적화합니다:
- `"document"` — 긴 문서의 핵심 내용을 잘 표현
- `"query"` — 짧은 검색 의도를 잘 표현

이 구분으로 **비대칭 검색** (짧은 쿼리 → 긴 문서) 정확도가 향상됩니다.

In [ ]:
# input_type 비교 실험
query_text = "최소 철근비"
doc_text = "RC 보의 최소 철근비는 0.25*sqrt(fck)/fy 이상이어야 하며, 1.4/fy 이상이어야 한다."

# query로 임베딩한 쿼리 vs document로 임베딩한 문서 (올바른 방법)
q_emb_correct = generate_embedding(query_text, input_type="query")
d_emb_correct = generate_embedding(doc_text, input_type="document")
sim_correct = cosine_similarity(q_emb_correct, d_emb_correct)

# 둘 다 document로 임베딩 (비추천)
q_emb_wrong = generate_embedding(query_text, input_type="document")
sim_wrong = cosine_similarity(q_emb_wrong, d_emb_correct)

print(f"올바른 방법 (query/document): {sim_correct:.4f}")
print(f"비추천 방법 (document/document): {sim_wrong:.4f}")

## 정리

- 텍스트를 벡터로 변환하면 **의미적 유사성**을 수치로 측정할 수 있다
- VoyageAI의 `input_type`을 구분하면 비대칭 검색 정확도가 향상된다
- Cosine Similarity로 쿼리-문서 유사도를 계산하고 상위 K개를 반환한다

다음 노트북에서는 **ChromaDB 벡터 DB**를 사용하여 대규모 검색을 구현합니다. → `S4_03_vector_search.ipynb`